# Chapter 3: Spark Sessions, Execution, and Joins

## Setup

This cell creates the `SparkSession` used throughout the notebook and two small DataFrames for transformation and join examples. `local[*]` runs Spark on this machine using available processor cores; in a cluster, the environment typically configures the cluster manager instead. The small shuffle partition setting is only for these teaching examples.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, col

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Spark Sessions, Execution, and Joins")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

sales = spark.createDataFrame(
    [
        (101, "Ava", "books", 120.0),
        (102, "Ben", "games", 75.0),
        (103, "Ava", "games", 210.0),
        (104, "Chen", "books", 90.0),
    ],
    ["order_id", "customer", "category", "amount"],
)

customers = spark.createDataFrame(
    [("Ava", "North"), ("Ben", "South"), ("Chen", "West")],
    ["customer", "region"],
)

sales.show()

## SparkSession

`SparkSession` is the main entry point for Spark SQL and DataFrame work. It lets an application read data, create DataFrames, run SQL, and access Spark functionality. The setup cell used `SparkSession.builder` to configure a session and `getOrCreate()` to reuse an existing session in the process or create one if needed.

In [ ]:
print("Application:", spark.sparkContext.appName)
print("Spark version:", spark.version)
sales.select("customer", "amount").show()

## Lazy evaluation and actions

**Lazy evaluation** is Spark's execution model in which transformations are not computed when they are defined. Instead, Spark records the requested work in a plan and waits until an action needs a result. For example, calling `filter()` creates a new DataFrame description; by itself, it does not read and filter all the rows immediately.

A **transformation** is an operation that defines a new DataFrame from an existing one. Transformations such as `filter`, `select`, and `groupBy` are generally lazy: they build up the plan rather than immediately return the processed rows.

| Transformation | Action |
| --- | --- |
| Defines a new DataFrame or RDD from existing data. | Requests a result or output from a DataFrame or RDD. |
| Usually returns another lazy DataFrame or RDD. | Returns a result to the driver, displays rows, or writes output to storage. |
| Examples: `filter`, `select`, `map`, `groupBy`. | Examples: `show`, `count`, `collect`, `write`. |
| Generally adds to the plan without starting computation. | Triggers execution of the required plan. |

An **action** is an operation that asks Spark to produce or expose a result. Examples include `show()` (display rows), `count()` (return a row count), `collect()` (return all rows to the driver), and `write` (save rows to an external destination). An action triggers Spark to analyze and execute the required plan, usually by starting one or more jobs. Some actions return a small result to the driver; `write` sends the result to storage. Avoid `collect()` on large datasets because all returned rows must fit in driver memory.

Because work is deferred, Spark can optimize multiple transformations together before execution. This is useful for performance, but it also means that some data-source or computation errors appear only when an action runs. If the same uncached DataFrame is used by multiple actions, Spark may execute its plan again for each action.

In [ ]:
# These transformations define a plan; they do not display or return the rows yet.
filtered_sales = sales.filter(col("amount") > 100)
selected_sales = filtered_sales.select("customer", "amount")

# Action: executes the plan and displays the matching rows.
selected_sales.show()

# Another action: computes and returns a count.
matching_rows = selected_sales.count()
print("Matching rows:", matching_rows)

## Query plans and Spark UI

Spark turns DataFrame operations into a query plan, optimizes it, and chooses physical operators to run it. `explain()` helps inspect those planned operations, including scans, filters, exchanges (shuffle boundaries), aggregations, and join strategies. The plan may not show every runtime detail, and adaptive query execution can change parts of the physical plan while a query runs.

The Spark UI shows runtime behavior. The **Jobs** tab shows jobs triggered by actions; **Stages** and their tasks show progress and performance; **SQL** displays DataFrame/SQL execution details; and **Executors** shows executor activity and storage. The UI is usually at `http://<driver-host>:4040` while the application is running. For completed applications, event logging and the History Server can preserve the UI.

In [ ]:
query = (
    sales.filter(col("amount") > 75)
    .groupBy("category")
    .sum("amount")
)

query.explain("formatted")
query.show()  # Runs the query; inspect its job in the Spark UI

## Spark RDD

An **RDD** (Resilient Distributed Dataset) is Spark's lower-level distributed collection. It is immutable, split into partitions, and processed in parallel. Spark records the transformations used to build an RDD (its lineage), which can allow lost partitions to be recomputed. RDD transformations such as `map` and `filter` are lazy; actions such as `sum`, `count`, and `collect` trigger computation. For structured data, DataFrames are often preferable because their schema and higher-level operations let Spark SQL optimize the work.

In [ ]:
numbers = spark.sparkContext.parallelize([1, 2, 3, 4], 2)
doubled_numbers = numbers.map(lambda value: value * 2)

print("Sum of doubled values:", doubled_numbers.sum())

## Narrow and wide transformations

A **narrow transformation** computes each output partition from a small, known set of input partitions. A **wide transformation** may need records from many partitions to be redistributed and combined; this redistribution is called a **shuffle**.

| Narrow transformation | Wide transformation |
| --- | --- |
| Each output partition depends on a limited set of input partitions. | An output partition may depend on records from many input partitions. |
| Usually does not require a shuffle. | Usually requires a shuffle to redistribute data. |
| Lower communication overhead; tasks can often run independently. | Higher cost from network transfer, disk I/O, and coordination. |
| Examples: `filter`, `map`, `select`. | Examples: `groupBy`, `reduceByKey`, repartitioning, many joins. |

For example, filtering can be applied to each partition independently. Grouping by a key must bring matching keys together, so Spark may shuffle records across the cluster. Shuffle dependencies commonly mark boundaries between execution stages.

In [ ]:
narrow_result = sales.filter(col("amount") > 75).select("customer", "amount")
wide_result = sales.groupBy("category").sum("amount")

print("Narrow transformation result:")
narrow_result.show()
print("Grouping result (requires data to be combined by category):")
wide_result.show()

## Repartition() versus Coalesce()

Both methods change the partition count, but differ in how they move data:

| `repartition(n)` | `coalesce(n)` |
| --- | --- |
| Can increase or decrease the partition count. | Primarily used to decrease the partition count. |
| Shuffles and redistributes data. | Usually reduces partitions with less data movement. |
| Can redistribute data more evenly. | May leave partitions uneven. |
| Use when more parallelism or balanced distribution is needed. | Use when fewer output partitions/files are wanted without a full shuffle. |

Changing partition count is not automatically faster—consider data size, skew, available cores, and output-file needs.

In [ ]:
print("Original partitions:", sales.rdd.getNumPartitions())

redistributed = sales.repartition(4)
reduced = redistributed.coalesce(2)

print("After repartition:", redistributed.rdd.getNumPartitions())
print("After coalesce:", reduced.rdd.getNumPartitions())

## Jobs, stages, and tasks

A **job** is started when an action requests a result, such as `count()` or a write. Spark divides a job into **stages**, commonly separated by shuffle boundaries. Each stage is divided into **tasks**, usually one per partition, and executors run those tasks.

| Term | Definition and role |
| --- | --- |
| **Job** | The complete computation needed for an action. It contains one or more stages; for example, a job triggered by `result.write.parquet(...)`. |
| **Stage** | A group of tasks that can run without crossing a shuffle boundary. A job is split into stages, commonly at shuffle boundaries. |
| **Task** | Work on one partition within a stage. Multiple tasks can run in parallel; for example, each task may filter one input partition. |

In short: **an action triggers a job; a job contains stages; stages contain tasks**. The exact number depends on the query and execution plan.

In [ ]:
stage_example = (
    sales.filter(col("amount") > 75)
    .groupBy("category")
    .sum("amount")
)

# This action starts a job. Inspect Jobs, Stages, and Tasks in the Spark UI.
stage_example.show()

## Shuffle joins

When neither input is small enough to broadcast, Spark may redistribute both datasets so rows with the same join key meet on the same partitions. This **shuffle join** can handle large inputs, but moving and possibly sorting data across the network costs time and disk I/O. A sort-merge join is a common strategy for large inputs. Filter rows and select needed columns before joining when possible, and inspect the plan and Spark UI for shuffle volume and skew.

In [ ]:
large_side = sales.repartition(4, "customer")
lookup_side = customers.repartition(4, "customer")

shuffle_join = large_side.join(lookup_side, on="customer", how="inner")
shuffle_join.explain("formatted")
shuffle_join.show()

## Broadcast joins

If one join input is small enough to fit safely in executor memory, Spark can send a copy of it to each executor. The larger input can then be joined locally without shuffling both inputs. Spark may choose a broadcast automatically using size estimates and configuration; `broadcast()` can also hint which DataFrame to broadcast. Do not broadcast a large dataset: copying it to executors can cause memory pressure or failures.

### Shuffle join versus broadcast join

| Shuffle join | Broadcast join |
| --- | --- |
| Redistributes join-key records across the cluster. | Copies the small side to executors; avoids shuffling both sides. |
| Best when both inputs are large or neither is safe to broadcast. | Best when one input is small enough to fit safely in executor memory. |
| Network and disk I/O are key costs; skew can make some tasks slow. | Memory pressure is a risk if the broadcast side is too large. |
| Often uses a sort-merge join for large equi-join inputs. | Often uses a broadcast hash join for eligible equi-joins. |

Neither strategy is always faster. Inspect the physical plan and Spark UI to confirm the strategy Spark selected.

In [ ]:
broadcast_join = sales.join(
    broadcast(customers),
    on="customer",
    how="inner",
)

broadcast_join.explain("formatted")
broadcast_join.show()